# 11장 보안 실습 — 웹 로그와 제한된 배치


## Goal

요청·상태를 분리하고 두 로컬 작업을 검증합니다.

[교안과 분석 질문](../../11-parallel-jobs/11-3-web-log-analysis.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-11-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'access.log': '192.0.2.10 - - [10/Sep/2026:09:01:00 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n192.0.2.10 - - [10/Sep/2026:09:01:10 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n198.51.100.8 - - [10/Sep/2026:09:01:20 +0900] "POST /login HTTP/1.1" 401 120 "-" "CourseBrowser/1.0"\n203.0.113.7 - - [10/Sep/2026:09:04:00 +0900] "GET /health HTTP/1.1" 200 12 "-" "CourseMonitor/1.0"\n203.0.113.7 - - [10/Sep/2026:09:04:10 +0900] "GET /uploads/report.txt HTTP/1.1" 200 45 "-" "CourseBrowser/1.0"\n192.0.2.10 - - [10/Sep/2026:09:04:20 +0900] "GET /admin HTTP/1.1" 404 80 "-" "CourseBrowser/1.0"\n', 'auth.log': '2026-09-10T09:01:00+09:00 lab-web-01 sshd[101]: Failed password for invalid user guest from 192.0.2.10 port 50100 ssh2\n2026-09-10T09:01:10+09:00 lab-web-01 sshd[102]: Failed password for analyst from 192.0.2.10 port 50101 ssh2\n2026-09-10T09:01:20+09:00 lab-web-01 sshd[103]: Failed password for analyst from 198.51.100.8 port 50102 ssh2\n2026-09-10T09:02:00+09:00 lab-web-01 sshd[104]: Accepted publickey for analyst from 192.0.2.10 port 50103 ssh2\n2026-09-10T09:03:00+09:00 lab-web-01 sudo: analyst : TTY=pts/0 ; PWD=/home/analyst ; USER=root ; COMMAND=/usr/bin/id\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: HTTP401=3, 200=2, 404=1; auth5행/access6행

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. 인용된 요청과 상태 분리


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '"' '{split($3, status, " "); print $2 "|" status[1]}' "$COURSE_DATA/access.log" > "$COURSE_OUT/requests.psv"
test "$(wc -l < "$COURSE_OUT/requests.psv")" -eq 6
grep -Fx 'GET /uploads/report.txt HTTP/1.1|200' "$COURSE_OUT/requests.psv"


### 2. 상태 분포와 요청 경로 해석


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' '{print $2}' "$COURSE_OUT/requests.psv" | LC_ALL=C sort | uniq -c > "$COURSE_OUT/status-counts.txt"
grep -Eq '^[[:space:]]*3 401$' "$COURSE_OUT/status-counts.txt"
grep -Eq '^[[:space:]]*2 200$' "$COURSE_OUT/status-counts.txt"
grep -Eq '^[[:space:]]*1 404$' "$COURSE_OUT/status-counts.txt"
cat "$COURSE_OUT/status-counts.txt"


### 3. 두 로컬 작업의 종료 상태 회수


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
wc -l < "$COURSE_DATA/auth.log" > "$COURSE_OUT/01.count" &
first_pid=$!
wc -l < "$COURSE_DATA/access.log" > "$COURSE_OUT/02.count" &
second_pid=$!
wait "$first_pid"
wait "$second_pid"
test "$(cat "$COURSE_OUT/01.count")" -eq 5
test "$(cat "$COURSE_OUT/02.count")" -eq 6
printf 'auth_rows=5 access_rows=6 order=fixed\n'


## Red Team ↔ Blue Team 사례 분석

### 사례: 업로드 경로의 200 응답과 호스트 영향

**Red Team 질문:** 공개된 웹 기능과 인증·파일 처리의 경계가 의도대로 구성되어 있는가? 경로와 상태 코드는 표면 동작의 단서이며 실제 파일 실행이나 권한 획득을 보여주지 않습니다. 실습은 합성 요청 로그 해석입니다.

| 연결 단계 | 분석 내용 |
|---|---|
| Goal / Boundary | 인증·자원 접근·파일 처리 경계의 위험 가설 검토 |
| Command / Observation | awk로 요청과 상태 분리. /uploads/report.txt 200은 접근 로그의 관찰 |
| System Change / Artifact | 요청 기록·앱 메시지·파일·프로세스 변화가 있을 수 있으나 각각 별도 확인 필요 |
| Log prerequisite | 웹 형식·프록시 주소 신뢰·앱 인증/오류·요청 ID·호스트 실행 자료의 수집 여부 |
| Blue Team Investigation | 웹 시각/요청 ID→앱 처리→파일 신원·생성→프로세스·통신 근거를 비교 |
| Detection | 상태 분포와 비정상 접근 문맥, 실제 호스트 영향의 일치를 단계별로 검토 |
| Mitigation | 원인이 확인된 인증·파일 저장/실행 분리·입력 처리·패치 정책을 개선 |

**반례와 해설:** 정상 보고서 다운로드도 같은 경로의 200을 남길 수 있습니다. SQL Injection·Path Traversal·WebShell이라는 이름을 요청 패턴에 붙였더라도 시도와 성공은 구분해야 합니다. 현재 자료에는 원문 요청 본문·DB 감사·파일 내용이 없어 해당 영향은 미확인입니다.

**제출 과제:** 401 세 건을 설명하는 공격 목적 가설과 정상 재시도 가설을 각각 적습니다. /uploads/report.txt에 대해서는 접근 사실, 부족한 파일 근거, 필요한 호스트 근거를 분리합니다. 로그 처리 작업 두 개의 성공과 보안 가설의 성공을 혼동하지 않으면 통과입니다.


## 역할별 분석 기록

같은 실행 결과로 아래 항목을 작성하고 상대 관점에서 검토합니다. 자동 테스트는 계산과 원본 보존만 확인하며 이 서술 과제는 강사 또는 동료 검토 대상입니다.

| 항목 | 학생 작성 |
|---|---|
| Red Team 목적·필요 조건 | 관찰에서 도출한 질문과 전제 |
| 실제 관찰 | 파일·행·이벤트 ID와 출력 |
| Artifact·로깅 전제 | 확보한 자료와 필요한 기록 기능 |
| Blue Team 조사 | 정상 반례·추가 근거·수집 한계 |
| 탐지·완화 | 필요한 필드·오탐 사례·확인된 원인에 맞는 조치 |


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
